<a href="https://colab.research.google.com/github/sevaradevoloper/machine-learning-class-work/blob/main/netflixtv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CatBoost

In [ ]:
# Dataset tanlash va yuklab olish

In [ ]:
!pip install opendatasets legacy-cgi --upgrade --quiet


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.3 MB/s eta 0:00:00


In [ ]:
import opendatasets as od

In [ ]:
od.download("https://www.kaggle.com/datasets/shivamb/netflix-shows")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: sevaradev
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/shivamb/netflix-shows


100%|██████████| 1.34M/1.34M [00:00<00:00, 52.1MB/s]

In [ ]:
import pandas as pd




In [ ]:
df = pd.read_csv("/content/netflix-shows/netflix_titles.csv")

In [ ]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


show_id — ID raqami (kino tartib raqami).

type — Turi (Kino yoki Serial).

title — Nomi (kino yoki serial nomi).

director — Rejissyor (film ijodkori).

cast — Aktyorlar (bosh roldagilar).

country — Davlat (kino olingan mamlakat).

date_added — Qo'shilgan sana (Netflix'ga yuklangan kun).

release_year — Chiqqan yili (ekranga chiqqan yili).

rating — Yosh cheklovi (kimlar ko'rishi mumkinligi).

duration — Davomiyligi (vaqti: daqiqa yoki fasl).

listed_in — Janri (kino qaysi turga kirishi).

description — Qisqacha tavsif (kino mazmuni).

In [ ]:
print(df.shape)

(8807, 12)


In [ ]:
df.columns.tolist()

['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in',
 'description']

In [ ]:
df["type"].value_counts()

# imbalanced target

,count
type,
Movie,6131
TV Show,2676


In [ ]:
# EDA (datasetni tanish) — qanday ustunlar bor, qaysilari kategorik

duration — Movie uchun "90 min", TV Show uchun "2 Seasons" deb yozilgan. Bu to'g'ridan-to'g'ri type'ni aytib beradi! → Bu ustunni tashlab yuborishimiz kerak (yoki keyinroq alohida ehtiyotkorlik bilan ishlash).

show_id, title, description, cast, director — bular deyarli har qatorda noyob (unique) qiymatlar. Ular model uchun foyda bermaydi va CatBoost'ni chalg'itadi. Bularni ham tashlaймiz.


Qoladigan foydali ustunlar (feature'lar):

country (kategorik) ✅
rating (kategorik — TV-MA, PG va h.k.) ✅
listed_in (kategorik — janrlar) ✅
release_year (raqamli) ✅
date_added'dan yil/oy ajratib olsa bo'ladi (keyinroq, ixtiyoriy)

In [ ]:
drop_col = ['show_id',
 'title',
 'director',
 'cast',
 'duration',
 'description']

In [ ]:
df = df.drop(columns=df[drop_col])
df

,type,country,date_added,release_year,rating,listed_in
0,Movie,United States,"September 25, 2021",2020,PG-13,Documentaries
1,TV Show,South Africa,"September 24, 2021",2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries"
2,TV Show,NaN,"September 24, 2021",2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
3,TV Show,NaN,"September 24, 2021",2021,TV-MA,"Docuseries, Reality TV"
4,TV Show,India,"September 24, 2021",2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."
...,...,...,...,...,...,...
8802,Movie,United States,"November 20, 2019",2007,R,"Cult Movies, Dramas, Thrillers"
8803,TV Show,NaN,"July 1, 2019",2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies"
8804,Movie,United States,"November 1, 2019",2009,R,"Comedies, Horror Movies"
8805,Movie,United States,"January 11, 2020",2006,PG,"Children & Family Movies, Comedies"


In [ ]:
# isnull
df.isnull().sum()

,0
type,0
country,831
date_added,10
release_year,0
rating,4
listed_in,0


In [ ]:
cat_cl = ['date_added','country','rating']
cat_mode = df[cat_cl].mode().iloc[0]
df[cat_cl] = df[cat_cl].fillna(cat_mode)
df.isnull().sum()

,0
type,0
country,0
date_added,0
release_year,0
rating,0
listed_in,0


In [ ]:
df = df.drop(columns=['date_added'])
df

,type,country,release_year,rating,listed_in
0,Movie,United States,2020,PG-13,Documentaries
1,TV Show,South Africa,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries"
2,TV Show,United States,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
3,TV Show,United States,2021,TV-MA,"Docuseries, Reality TV"
4,TV Show,India,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."
...,...,...,...,...,...
8802,Movie,United States,2007,R,"Cult Movies, Dramas, Thrillers"
8803,TV Show,United States,2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies"
8804,Movie,United States,2009,R,"Comedies, Horror Movies"
8805,Movie,United States,2006,PG,"Children & Family Movies, Comedies"


In [ ]:
# X, y ajratish + kategorik ustunlarni belgilash + train/test split

df["type"] = df["type"].replace({'Movie': 1,'TV Show': 0})
df

/tmp/ipykernel_1987/3342547742.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["type"] = df["type"].replace({'Movie': 1,'TV Show': 0})


,type,country,release_year,rating,listed_in
0,1,United States,2020,PG-13,Documentaries
1,0,South Africa,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries"
2,0,United States,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
3,0,United States,2021,TV-MA,"Docuseries, Reality TV"
4,0,India,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ..."
...,...,...,...,...,...
8802,1,United States,2007,R,"Cult Movies, Dramas, Thrillers"
8803,0,United States,2018,TV-Y7,"Kids' TV, Korean TV Shows, TV Comedies"
8804,1,United States,2009,R,"Comedies, Horror Movies"
8805,1,United States,2006,PG,"Children & Family Movies, Comedies"


In [ ]:
# X va y
X = df.drop(columns=['type'])
y = df['type']

In [ ]:
# cat_features
cat_features = X.select_dtypes(include='object').columns.tolist()
cat_features

['country', 'rating', 'listed_in']

In [ ]:
# train_test split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state = 42,stratify= y)
print(f'X_train shape {X_train.shape}')
print(f'X_test shape:{X_test.shape}')
print(f'y_train shape {y_train.shape}')
print(f'y_test shape:{y_test.shape}')

X_train shape (7045, 4)
X_test shape:(1762, 4)
y_train shape (7045,)
y_test shape:(1762,)


In [ ]:
# CatBoost bilan train
from catboost import CatBoostClassifier

In [ ]:
import time

In [ ]:
start = time.perf_counter()
model = CatBoostClassifier(
    iterations= 300,
    learning_rate = 0.1,
    depth = 6,
    random_state=42,
    verbose = 100
)

model.fit(X_train,y_train,cat_features = cat_features)

0:	learn: 0.4060501	total: 13.5ms	remaining: 4.04s
100:	learn: 0.0258537	total: 1.17s	remaining: 2.3s
200:	learn: 0.0230836	total: 2.04s	remaining: 1s
299:	learn: 0.0209859	total: 2.79s	remaining: 0us


CatBoostClassifier(depth=6, iterations=300, learning_rate=0.1, random_state=42, verbose=100)

In [ ]:
catboost_time = time.perf_counter() - start
print(f'Train uchun olingan vaqt : {catboost_time}')

Train uchun olingan vaqt : 2.896783527000025


In [ ]:
# predict
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score


In [ ]:
# baholash

acc = accuracy_score(y_test,y_pred)
print(f'Accurancy Catgboost score {acc}')

Accurancy Catgboost score 0.9977298524404086


In [ ]:
model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,listed_in,60.615810
1,rating,17.060122
2,country,12.189776
3,release_year,10.134292


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
cat_col = ['country', 'rating', 'listed_in']

In [ ]:
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_train_enc[cat_col] = encoder.fit_transform(X_train_enc[cat_col])

X_test_enc[cat_col] = encoder.transform(X_test_enc[cat_col])

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators = 300,
    learning_rate = 0.1,
    max_depth = 6,
    random_state = 42
)

start_time = time.perf_counter()
lgbm.fit(X_train_enc,y_train)
lgbm_time = time.perf_counter() - start_time

y_pred_lgbm = lgbm.predict(X_test_enc)
lgbm_accuracy = accuracy_score(y_test,y_pred_lgbm)
print(f"LGBM Accuracy: {lgbm_accuracy:.4f}")
print(f"LGBM Train time: {lgbm_time:.4f} soniya")

[LightGBM] [Info] Number of positive: 4904, number of negative: 2141
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000179 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 557
[LightGBM] [Info] Number of data points in the train set: 7045, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.696097 -> initscore=0.828778
[LightGBM] [Info] Start training from score 0.828778
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

In [ ]:
# Birlashtrilgan df

analiz = pd.DataFrame(
    {
        "Accuracy": [acc, lgbm_accuracy],
        "Train Time (sec)": [catboost_time, lgbm_time],
        "Code Lines": [4,8],
        "Encoding Kerakmi?": ["Yo'q (Avtomatik)", "Ha (Qo'lda)"],
    },
    index=["CatBoost", "LightGBM"],
)

# Natijani ko'rish
print(analiz)


          Accuracy  Train Time (sec)  Code Lines Encoding Kerakmi?
CatBoost  0.997730          2.896784           4  Yo'q (Avtomatik)
LightGBM  0.990919          0.444285           8       Ha (Qo'lda)


In [ ]:
# paramsni sozlab koramiz

cat_cols = ["country", "rating", "listed_in"]

variants = [
    {"depth": 6, "learning_rate": 0.1, "iterations": 300},
    {"depth": 10, "learning_rate": 0.1, "iterations": 300},
    {"depth": 6, "learning_rate": 0.03, "iterations": 500},
    {"depth": 8, "learning_rate": 0.3, "iterations": 200},
]

results_list = []

for params in variants:

    model = CatBoostClassifier(**params, random_seed=42, verbose=False)

    start_time = time.perf_counter()
    model.fit(X_train, y_train, cat_features=cat_cols)
    train_time = time.perf_counter() - start_time

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    results_list.append(
        {
            "depth": params["depth"],
            "learning_rate": params["learning_rate"],
            "iterations": params["iterations"],
            "accuracy": acc,
            "train_time": train_time,
        }
    )

df_results = pd.DataFrame(results_list)
df_results = df_results.sort_values("accuracy", ascending=False)

# Natijani ko'rish
print(df_results)



   depth  learning_rate  iterations  accuracy  train_time
0      6           0.10         300  0.997730    3.145192
1     10           0.10         300  0.997162    8.319662
3      8           0.30         200  0.997162    5.542291
2      6           0.03         500  0.996595    6.334215


XULOSA: Ushbu amaliy vazifada jadval ko'rinishidagi ma'lumotlarni tasniflash (classification) masalasi CatBoost va LightGBM algoritmlari yordamida o'rganildi. Tajriba natijasida matnli (kategorik) ustunlar bilan ishlashda CatBoost o'zining avtomatik encoding tizimi tufayli LightGBM-ga qaraganda 2 barobar kamroq kod yozilishini ta'minladi va yuqori qulaylik ko'rsatdi. Biroq, tezlik nuqtai nazaridan LightGBM modeli qo'lda o'tkazilgan Ordinal Encoding yordamida CatBoost-dan sezilarli darajada tezroq (0.44s vs 2.90s) ishladi. Giperparametrlarni tuning qilish jarayonida aniqlandi-ki, model chuqurligini (depth=10) yoki iteratsiyalar sonini (iterations=500) oshirish aniqlikni yaxshilamaydi, balki o'qitish vaqtini asossiz ravishda uzaytirib yuboradi. Ma'lumotlar to'plamidagi kuchli target leakage (sinf belgilarining xususiyatlar ichida qolib ketishi) tufayli har ikkala model ham 99% dan yuqori ideal aniqlikka erishdi, ammo amaliy jihatdan eng maqbul muvozanatni depth=6 bo'lgan asosiy CatBoost modeli ko'rsatdi.